# RiskCheck AI - Automated Retraining

This notebook manages controlled retraining of the RiskCheck AI fraud detection model using investigator-verified outcomes.

### Retraining workflow
1. Load new investigator-verified labels.
2. Combine verified outcomes with the approved training data.
3. Train a candidate model using the same 32 model features.
4. Evaluate the candidate against a controlled holdout dataset.
5. Compare candidate performance with the current production model.
6. Promote the candidate only if the required performance criteria are met.
7. Mark successfully processed verified labels as consumed.

**Important:** Model predictions are never used as training labels. Only investigator-verified outcomes are eligible for retraining.

In [0]:
feedback_table = "workspace.default.riskcheck_investigator_feedback"

verified_feedback_df = spark.sql(f"""
    SELECT
        invoice_id,
        verified_label,
        investigator_note,
        verified_at
    FROM {feedback_table}
    WHERE consumed_by_training = false
""")

print(f"New verified labels available: {verified_feedback_df.count()}")

display(verified_feedback_df)

New verified labels available: 0


invoice_id,verified_label,investigator_note,verified_at


In [0]:
MIN_VERIFIED_LABELS = 200

new_verified_count = verified_feedback_df.count()

if new_verified_count < MIN_VERIFIED_LABELS:
    print(
        f"Retraining skipped: only {new_verified_count} "
        f"new verified labels available. "
        f"Minimum required: {MIN_VERIFIED_LABELS}."
    )

    dbutils.notebook.exit(
        f"SKIPPED - {new_verified_count}/{MIN_VERIFIED_LABELS} verified labels"
    )

In [0]:
scored_table = "workspace.default.riskcheck_scored_invoices"

scored_invoices_df = spark.table(scored_table)

print("Scored invoice table loaded successfully.")
print(f"Rows: {scored_invoices_df.count():,}")
print(f"Columns: {len(scored_invoices_df.columns)}")

Scored invoice table loaded successfully.
Rows: 100
Columns: 42


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

latest_window = (
    Window
    .partitionBy("invoice_id")
    .orderBy(F.col("scored_at").desc())
)

latest_scored_df = (
    scored_invoices_df
    .withColumn("row_num", F.row_number().over(latest_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

print(f"Latest unique scored invoices: {latest_scored_df.count():,}")

display(
    latest_scored_df.select(
        "invoice_id",
        "invoice_amount",
        "invoice_amount_zscore",
        "supplier_risk_score",
        "scored_at"
    )
)

Latest unique scored invoices: 10


invoice_id,invoice_amount,invoice_amount_zscore,supplier_risk_score,scored_at
INV-DEMO-001,420.0,-0.35,0.08,2026-09-09T22:05:59.331Z
INV-DEMO-002,1250.0,0.22,0.14,2026-09-09T22:06:09.546Z
INV-DEMO-003,3100.0,1.15,0.42,2026-09-09T22:05:55.347Z
INV-DEMO-004,6900.0,2.75,0.67,2026-09-09T22:05:51.712Z
INV-DEMO-005,9800.0,3.9,0.89,2026-09-09T22:05:34.583Z
INV-DEMO-006,560.0,-0.18,0.05,2026-09-09T22:06:06.094Z
INV-DEMO-007,4200.0,2.1,0.58,2026-09-09T22:06:02.880Z
INV-DEMO-008,15000.0,4.4,0.95,2026-09-09T22:05:44.314Z
INV-DEMO-009,2100.0,0.12,0.2,2026-09-09T22:06:13.610Z
INV-DEMO-010,7600.0,3.2,0.78,2026-09-09T22:05:48.305Z


In [0]:
final_feature_columns = [
    "invoice_amount",
    "submission_hour",
    "supplier_invoice_count_30d",
    "supplier_avg_amount_90d",
    "invoice_amount_zscore",
    "duplicate_invoice_flag",
    "split_invoice_flag",
    "late_night_submission_flag",
    "supplier_age_days",
    "supplier_risk_score",
    "blacklisted_flag",
    "avg_invoice_amount",
    "annual_budget",
    "ocr_total_extracted",
    "image_tamper_flag",
    "amount_to_supplier_90d_avg_ratio",
    "amount_to_supplier_avg_ratio",
    "amount_diff_supplier_90d_avg",
    "supplier_age_years",
    "invoice_day_of_week",
    "weekend_invoice_flag",
    "outside_business_hours_flag",
    "invoice_to_department_budget_ratio",
    "log_invoice_amount",
    "image_metadata_available_flag",
    "invoice_ocr_amount_diff",
    "invoice_ocr_relative_diff",
    "payment_terms_NET30",
    "payment_terms_NET60",
    "payment_terms_NET90",
    "invoice_type_GOODS",
    "invoice_type_SERVICES"
]

print(f"Model features defined: {len(final_feature_columns)}")

Model features defined: 32


In [0]:
verified_new_training_df = (
    verified_feedback_df.alias("f")
    .join(
        latest_scored_df.alias("s"),
        F.col("f.invoice_id") == F.col("s.invoice_id"),
        "inner"
    )
    .select(
        F.col("f.invoice_id"),
        *[F.col(f"s.{c}") for c in final_feature_columns],
        F.col("f.verified_label").alias("is_fraud")
    )
)

print(
    f"Verified uploaded invoices available for training: "
    f"{verified_new_training_df.count():,}"
)

display(
    verified_new_training_df.select(
        "invoice_id",
        "invoice_amount",
        "invoice_amount_zscore",
        "supplier_risk_score",
        "is_fraud"
    )
)

Verified uploaded invoices available for training: 1


invoice_id,invoice_amount,invoice_amount_zscore,supplier_risk_score,is_fraud
INV-DEMO-005,9800.0,3.9,0.89,1


In [0]:
null_check = verified_new_training_df.select(
    [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in final_feature_columns
    ]
)

display(null_check)

invoice_amount,submission_hour,supplier_invoice_count_30d,supplier_avg_amount_90d,invoice_amount_zscore,duplicate_invoice_flag,split_invoice_flag,late_night_submission_flag,supplier_age_days,supplier_risk_score,blacklisted_flag,avg_invoice_amount,annual_budget,ocr_total_extracted,image_tamper_flag,amount_to_supplier_90d_avg_ratio,amount_to_supplier_avg_ratio,amount_diff_supplier_90d_avg,supplier_age_years,invoice_day_of_week,weekend_invoice_flag,outside_business_hours_flag,invoice_to_department_budget_ratio,log_invoice_amount,image_metadata_available_flag,invoice_ocr_amount_diff,invoice_ocr_relative_diff,payment_terms_NET30,payment_terms_NET60,payment_terms_NET90,invoice_type_GOODS,invoice_type_SERVICES
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
training_base_table = "workspace.default.riskcheck_training_base"

training_base_df = spark.table(training_base_table)

print("Training base loaded successfully.")
print(f"Rows: {training_base_df.count():,}")
print(f"Columns: {len(training_base_df.columns)}")

Training base loaded successfully.
Rows: 300,000
Columns: 35


In [0]:
historical_training_df = training_base_df.select(
    "invoice_id",
    *final_feature_columns,
    "is_fraud"
)

new_verified_training_df = verified_new_training_df.select(
    "invoice_id",
    *final_feature_columns,
    "is_fraud"
)

combined_training_df = historical_training_df.unionByName(
    new_verified_training_df
)

print(f"Historical training rows: {historical_training_df.count():,}")
print(f"New verified rows: {new_verified_training_df.count():,}")
print(f"Combined training rows: {combined_training_df.count():,}")

Historical training rows: 300,000
New verified rows: 1
Combined training rows: 300,001


In [0]:
verified_ids = [
    row["invoice_id"]
    for row in new_verified_training_df.select("invoice_id").collect()
]

print("Verified invoice IDs prepared for consumption:")
print(verified_ids)

Verified invoice IDs prepared for consumption:
['INV-DEMO-005']


In [0]:
%sql
SELECT
    invoice_id,
    verified_label,
    investigator_note,
    verified_at,
    consumed_by_training
FROM workspace.default.riskcheck_investigator_feedback
ORDER BY verified_at DESC;

invoice_id,verified_label,investigator_note,verified_at,consumed_by_training
INV-DEMO-005,1,Confirmed fraudulent during investigation,2026-09-09T22:27:53.341Z,false


In [0]:
if verified_ids:
    ids_sql = ", ".join([f"'{x}'" for x in verified_ids])

    spark.sql(f"""
        UPDATE workspace.default.riskcheck_investigator_feedback
        SET consumed_by_training = true
        WHERE invoice_id IN ({ids_sql})
          AND consumed_by_training = false
    """)

    print("Verified labels marked as consumed.")
else:
    print("No verified labels to mark as consumed.")

Verified labels marked as consumed.


In [0]:
display(
    spark.sql("""
        SELECT
            invoice_id,
            verified_label,
            consumed_by_training
        FROM workspace.default.riskcheck_investigator_feedback
        ORDER BY verified_at DESC
    """)
)

invoice_id,verified_label,consumed_by_training
INV-DEMO-005,1,true


In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Use the historical train split plus new verified rows
historical_train_df = (
    training_base_df
    .filter(F.col("split") == "train")
    .select(*final_feature_columns, "is_fraud")
)

historical_test_df = (
    training_base_df
    .filter(F.col("split") == "test")
    .select(*final_feature_columns, "is_fraud")
)

# Add only investigator-verified new rows to the training set
candidate_train_df = historical_train_df.unionByName(
    new_verified_training_df.select(*final_feature_columns, "is_fraud")
)

# Convert to pandas for sklearn
candidate_train_pdf = candidate_train_df.toPandas()
candidate_test_pdf = historical_test_df.toPandas()

X_candidate_train = candidate_train_pdf[final_feature_columns]
y_candidate_train = candidate_train_pdf["is_fraud"].astype(int)

X_candidate_test = candidate_test_pdf[final_feature_columns]
y_candidate_test = candidate_test_pdf["is_fraud"].astype(int)

print("Candidate training rows:", len(X_candidate_train))
print("Candidate test rows:", len(X_candidate_test))